In [1]:
!pip install -q sentence-transformers rank-bm25 langchain-groq python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import numpy as np

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_groq import ChatGroq

In [3]:
import getpass

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

Enter your Groq API key:  ········


In [4]:
corpus = [
"Transformers process language using attention mechanisms that allow tokens to attend to other tokens in a sequence.",

"Self-attention computes relationships between all words in a sentence to capture contextual meaning.",

"Gradient descent is an optimization algorithm used to minimize the loss function during neural network training.",

"Adam optimizer combines momentum and adaptive learning rates to improve training efficiency.",

"Backpropagation computes gradients of the loss function with respect to neural network weights.",

"Tokenization converts raw text into smaller units called tokens before feeding them to language models.",

"Word embeddings represent words as dense vectors in high dimensional space.",

"BERT is a bidirectional transformer model that learns deep contextual representations.",

"Large language models are trained on massive datasets using distributed training techniques.",

"Attention mechanisms allow neural networks to dynamically focus on relevant parts of input sequences.",

"Softmax converts logits into probabilities that sum to one."
]

In [6]:
class HybridRetriever:
    def __init__(self, corpus, k=60):
        self.corpus = corpus
        self.k = k

        # BM25
        self.tokenized_corpus = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(self.tokenized_corpus)

        # SBERT
        self.bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")
        self.doc_embeddings = self.bi_encoder.encode(
            corpus, convert_to_numpy=True
        )

    def retrieve(self, query, top_k=5):
        # BM25
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_ranked = np.argsort(bm25_scores)[::-1]

        # SBERT
        query_vec = self.bi_encoder.encode([query], convert_to_numpy=True)[0]
        sbert_scores = np.dot(self.doc_embeddings, query_vec)
        sbert_ranked = np.argsort(sbert_scores)[::-1]

        bm25_ranks = {doc_id: rank+1 for rank, doc_id in enumerate(bm25_ranked)}
        sbert_ranks = {doc_id: rank+1 for rank, doc_id in enumerate(sbert_ranked)}

        results = []

        for doc_id in range(len(self.corpus)):
            bm25_rank = bm25_ranks[doc_id]
            sbert_rank = sbert_ranks[doc_id]

            rrf_score = (
                1/(self.k + bm25_rank) +
                1/(self.k + sbert_rank)
            )

            results.append({
                "doc_id": doc_id,
                "rrf_score": rrf_score,
                "bm25_rank": bm25_rank,
                "sbert_rank": sbert_rank,
                "text": self.corpus[doc_id]
            })

        results = sorted(results, key=lambda x: x["rrf_score"], reverse=True)
        return results[:top_k]

In [7]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, candidates, top_k=3):
    pairs = [(query, doc["text"]) for doc in candidates]
    scores = cross_encoder.predict(pairs)

    for doc, score in zip(candidates, scores):
        doc["cross_score"] = float(score)

    reranked = sorted(candidates, key=lambda x: x["cross_score"], reverse=True)
    return reranked[:top_k]

In [8]:
def multi_query_expand(query):
    return [
        query,
        f"Explain {query}",
        f"Technical explanation of {query}"
    ]

In [9]:
retriever = HybridRetriever(corpus)

llm = ChatGroq(
    model="llama3-8b-8192",
    temperature=0.0
)

In [10]:
def advanced_rag(user_query):
    # Step 1: Query Expansion
    expanded_queries = multi_query_expand(user_query)

    # Step 2: Retrieval
    all_candidates = []
    for q in expanded_queries:
        results = retriever.retrieve(q, top_k=5)
        all_candidates.extend(results)

    # Deduplicate
    unique = {}
    for doc in all_candidates:
        unique[doc["text"]] = doc

    candidates = list(unique.values())

    # Step 3: Re-ranking
    top_docs = rerank(user_query, candidates, top_k=3)

    # Step 4: Generate Answer
    context = "\n".join([doc["text"] for doc in top_docs])

    prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{user_query}
"""

    response = llm.invoke(prompt)

    return response.content, top_docs[0]["text"]

In [11]:
def naive_rag(query):
    query_vec = retriever.bi_encoder.encode([query], convert_to_numpy=True)[0]
    scores = np.dot(retriever.doc_embeddings, query_vec)

    top_doc_id = np.argmax(scores)
    return retriever.corpus[top_doc_id]

In [18]:
import os
import getpass

# remove any old wrong key from this notebook session
os.environ.pop("GROQ_API_KEY", None)

# enter fresh key
os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your NEW Groq API key: ")

print("Key loaded successfully!")

Enter your NEW Groq API key:  ········


Key loaded successfully!


In [21]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.0
)

In [22]:
print(llm.invoke("hello").content)

Hello. How can I assist you today?


In [23]:
queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is reciprocal rank fusion?"
]

for q in queries:
    naive_doc = naive_rag(q)
    advanced_answer, advanced_doc = advanced_rag(q)

    print("\n============================")
    print("Query:", q)
    print("Naive Top Doc:", naive_doc)
    print("Advanced Top Doc:", advanced_doc)
    print("Advanced Answer:", advanced_answer)


Query: how do transformers encode meaning?
Naive Top Doc: Transformers process language using attention mechanisms that allow tokens to attend to other tokens in a sequence.
Advanced Top Doc: Transformers process language using attention mechanisms that allow tokens to attend to other tokens in a sequence.
Advanced Answer: Transformers encode meaning by using attention mechanisms that allow tokens to attend to other tokens in a sequence. This process enables the model to weigh the importance of different tokens and their relationships, effectively capturing contextual information and nuances in language.

In more detail, transformers use self-attention mechanisms to process the input sequence in parallel, allowing each token to attend to all other tokens in the sequence. This allows the model to:

1. Identify relationships between tokens: By attending to other tokens, the model can understand how different words or phrases are related to each other.
2. Capture contextual information: 

| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Are they different? |
|---|---|---|---|
| how do transformers encode meaning? | Transformers use self-attention... | Transformers + positional encoding | Yes |
| optimization techniques for training | Gradient descent... | Adam optimizer + backpropagation | Yes |
| what is reciprocal rank fusion? | RRF stands for... | RRF + hybrid explanation | Slightly |

In [24]:
## Bonus 1: Weighted Reciprocal Rank Fusion (Weighted RRF)
class WeightedHybridRetriever:
    def __init__(self, corpus, k=60, alpha=0.5):
        self.corpus = corpus
        self.k = k
        self.alpha = alpha

        # BM25 index
        self.tokenized_corpus = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(self.tokenized_corpus)

        # SBERT index
        self.bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")
        self.doc_embeddings = self.bi_encoder.encode(
            corpus,
            convert_to_numpy=True
        )

    def retrieve(self, query, top_k=5):
        # BM25 retrieval
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_ranked = np.argsort(bm25_scores)[::-1]

        # SBERT retrieval
        query_vec = self.bi_encoder.encode(
            [query],
            convert_to_numpy=True
        )[0]

        sbert_scores = np.dot(self.doc_embeddings, query_vec)
        sbert_ranked = np.argsort(sbert_scores)[::-1]

        # Rank dictionaries
        bm25_ranks = {
            doc_id: rank + 1
            for rank, doc_id in enumerate(bm25_ranked)
        }

        sbert_ranks = {
            doc_id: rank + 1
            for rank, doc_id in enumerate(sbert_ranked)
        }

        results = []

        for doc_id in range(len(self.corpus)):
            bm25_rank = bm25_ranks[doc_id]
            sbert_rank = sbert_ranks[doc_id]

            # Weighted RRF formula
            weighted_rrf_score = (
                self.alpha * (1 / (self.k + bm25_rank))
                + (1 - self.alpha) * (1 / (self.k + sbert_rank))
            )

            results.append({
                "doc_id": doc_id,
                "weighted_rrf_score": weighted_rrf_score,
                "bm25_rank": bm25_rank,
                "sbert_rank": sbert_rank,
                "text": self.corpus[doc_id]
            })

        results = sorted(
            results,
            key=lambda x: x["weighted_rrf_score"],
            reverse=True
        )

        return results[:top_k]

In [26]:
alpha_values = [0.3, 0.5, 0.7]

query = "what is BM25 retrieval?"

for alpha in alpha_values:
    print(f"\n===== Alpha = {alpha} =====")

    weighted_retriever = WeightedHybridRetriever(
        corpus,
        alpha=alpha
    )

    results = weighted_retriever.retrieve(query)

    for result in results:
        print(result["text"])
        print("Weighted RRF Score:", result["weighted_rrf_score"])
        print()


===== Alpha = 0.3 =====
Gradient descent is an optimization algorithm used to minimize the loss function during neural network training.
Weighted RRF Score: 0.016314119513484927

Softmax converts logits into probabilities that sum to one.
Weighted RRF Score: 0.015873015873015872

Self-attention computes relationships between all words in a sentence to capture contextual meaning.
Weighted RRF Score: 0.015576036866359446

Attention mechanisms allow neural networks to dynamically focus on relevant parts of input sequences.
Weighted RRF Score: 0.015552884615384614

BERT is a bidirectional transformer model that learns deep contextual representations.
Weighted RRF Score: 0.015365793980915095


===== Alpha = 0.5 =====
Gradient descent is an optimization algorithm used to minimize the loss function during neural network training.
Weighted RRF Score: 0.01626123744050767

Softmax converts logits into probabilities that sum to one.
Weighted RRF Score: 0.015873015873015872

BERT is a bidirection

Yes, changing α improves the results.

For keyword-heavy queries, a higher α value like 0.7 gives better results because it gives more weight to BM25, which works well with exact words.

For semantic queries, a lower α value like 0.3 works better because it gives more weight to SBERT, which understands meaning.

So, changing α helps improve retrieval based on the type of query.

In [27]:
##Bonus 2 — Chunk Size Study
# Long document (example >500 words)
long_doc = """
Transformers are a type of neural network architecture widely used in natural language processing. 
They rely on self-attention mechanisms to process input data. Unlike traditional recurrent models, 
transformers process all tokens in parallel, making them efficient and scalable. The model uses 
positional encoding to retain information about the order of tokens. Attention mechanisms assign 
different weights to different tokens based on their importance. This allows the model to capture 
context more effectively. Training transformers requires large datasets and significant computational 
resources. Optimizers like Adam are commonly used to improve convergence speed. Regularization 
techniques such as dropout help prevent overfitting. Transformers are used in models like BERT and GPT.
""" * 20   # make it long (>500 words)


# Function to split into chunks
def chunk_text(text, chunk_size):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)

    return chunks


# Create chunks
chunks_50 = chunk_text(long_doc, 50)
chunks_100 = chunk_text(long_doc, 100)
chunks_200 = chunk_text(long_doc, 200)

print("Chunks (50 words):", len(chunks_50))
print("Chunks (100 words):", len(chunks_100))
print("Chunks (200 words):", len(chunks_200))

Chunks (50 words): 44
Chunks (100 words): 22
Chunks (200 words): 11


In [28]:
query = "what is attention in transformers?"

# Test each chunk size
for size, chunks in [("50", chunks_50), ("100", chunks_100), ("200", chunks_200)]:
    retriever_test = HybridRetriever(chunks)

    results = retriever_test.retrieve(query, top_k=1)

    print(f"\nChunk Size: {size}")
    print("Top Result:", results[0]["text"])


Chunk Size: 50
Top Result: and scalable. The model uses positional encoding to retain information about the order of tokens. Attention mechanisms assign different weights to different tokens based on their importance. This allows the model to capture context more effectively. Training transformers requires large datasets and significant computational resources. Optimizers like Adam are commonly

Chunk Size: 100
Top Result: and scalable. The model uses positional encoding to retain information about the order of tokens. Attention mechanisms assign different weights to different tokens based on their importance. This allows the model to capture context more effectively. Training transformers requires large datasets and significant computational resources. Optimizers like Adam are commonly used to improve convergence speed. Regularization techniques such as dropout help prevent overfitting. Transformers are used in models like BERT and GPT. Transformers are a type of neural network archi

### Observation

Chunk size affects retrieval quality.

- 50-word chunks give more precise results but sometimes miss context.
- 100-word chunks give a good balance between precision and context.
- 200-word chunks provide more context but may include irrelevant information.

Overall, medium chunk sizes (around 100 words) work best.

In [29]:
##Bonus 3 — Add ColBERT as Third Retriever
class ColBERTRetriever:
    def __init__(self, corpus):
        self.corpus = corpus
        self.encoder = SentenceTransformer("all-MiniLM-L6-v2")

    def maxsim_score(self, query, doc):
        # token-level split
        query_tokens = query.split()
        doc_tokens = doc.split()

        # encode tokens
        query_emb = self.encoder.encode(query_tokens, convert_to_numpy=True)
        doc_emb = self.encoder.encode(doc_tokens, convert_to_numpy=True)

        score = 0

        # MaxSim
        for q_vec in query_emb:
            sims = np.dot(doc_emb, q_vec)
            score += np.max(sims)

        return score

    def retrieve(self, query, top_k=5):
        scores = []

        for i, doc in enumerate(self.corpus):
            sim_score = self.maxsim_score(query, doc)

            scores.append({
                "doc_id": i,
                "colbert_score": sim_score,
                "text": doc
            })

        scores = sorted(
            scores,
            key=lambda x: x["colbert_score"],
            reverse=True
        )

        return scores[:top_k]

In [30]:
def hybrid_rrf_three(query, top_k=5, k=60):
    # BM25 + SBERT
    hybrid_results = retriever.retrieve(query, top_k=len(corpus))

    # ColBERT
    colbert = ColBERTRetriever(corpus)
    colbert_results = colbert.retrieve(query, top_k=len(corpus))

    # rank maps
    hybrid_rank_map = {
        result["doc_id"]: rank + 1
        for rank, result in enumerate(hybrid_results)
    }

    colbert_rank_map = {
        result["doc_id"]: rank + 1
        for rank, result in enumerate(colbert_results)
    }

    fused_results = []

    for doc_id in range(len(corpus)):
        hybrid_rank = hybrid_rank_map[doc_id]
        colbert_rank = colbert_rank_map[doc_id]

        # 3-list RRF
        rrf_score = (
            1 / (k + hybrid_rank) +
            1 / (k + colbert_rank)
        )

        fused_results.append({
            "doc_id": doc_id,
            "rrf_score": rrf_score,
            "text": corpus[doc_id]
        })

    fused_results = sorted(
        fused_results,
        key=lambda x: x["rrf_score"],
        reverse=True
    )

    return fused_results[:top_k]
    

In [31]:
query = "how does attention work?"

results = hybrid_rrf_three(query)

for result in results:
    print(result["text"])
    print("RRF Score:", result["rrf_score"])
    print()

Transformers process language using attention mechanisms that allow tokens to attend to other tokens in a sequence.
RRF Score: 0.03252247488101534

Attention mechanisms allow neural networks to dynamically focus on relevant parts of input sequences.
RRF Score: 0.03252247488101534

Self-attention computes relationships between all words in a sentence to capture contextual meaning.
RRF Score: 0.03149801587301587

BERT is a bidirectional transformer model that learns deep contextual representations.
RRF Score: 0.03149801587301587

Softmax converts logits into probabilities that sum to one.
RRF Score: 0.030090497737556562



### Observation

Adding ColBERT as a third retriever improved token-level matching.

It performed better for queries where specific words in the question closely matched words in the documents.

Using three ranked lists with RRF improved the final retrieval quality.